In [ ]:
'''
"Given current state, predict next 100 actions"
# Future Prediction
input:  [pos_t0, pos_t1, ..., pos_t99]    # Current state sequence
output: [act_t100, act_t101, ..., act_t199] # Next 100 actions
'''
'''
predicting actions that correspond to input timesteps.
'''
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        return None, None, None, None

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

def load_all_episodes(data_dir):
    """
    Load all HDF5 files from the specified directory
    Args:
        data_dir: directory containing HDF5 files
    Returns:
        combined qpos, qvel, action arrays and list of image dicts
    """
    print(f"Loading episodes from: {data_dir}")
    
    # Find all HDF5 files
    hdf5_files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    hdf5_files.sort()  # Sort for consistent ordering
    
    if not hdf5_files:
        print("No HDF5 files found in the directory!")
        return None, None, None, None
    
    print(f"Found {len(hdf5_files)} HDF5 files: {hdf5_files}")
    
    all_qpos = []
    all_qvel = []
    all_action = []
    all_image_dicts = []
    
    episode_info = []
    
    for filename in hdf5_files:
        filepath = os.path.join(data_dir, filename)
        print(f"Loading {filename}...")
        
        qpos, qvel, action, image_dict = load_hdf5(filepath)
        
        if qpos is not None:
            print(f"  - Episode length: {len(qpos)} timesteps")
            print(f"  - QPos shape: {qpos.shape}")
            print(f"  - Action shape: {action.shape}")
            
            all_qpos.append(qpos)
            all_qvel.append(qvel)
            all_action.append(action)
            all_image_dicts.append(image_dict)
            
            episode_info.append({
                'filename': filename,
                'length': len(qpos),
                'start_idx': sum(len(ep) for ep in all_qpos[:-1]),
                'end_idx': sum(len(ep) for ep in all_qpos)
            })
        else:
            print(f"  - Failed to load {filename}")
    
    if not all_qpos:
        print("No valid episodes loaded!")
        return None, None, None, None
    
    # Concatenate all episodes
    combined_qpos = np.concatenate(all_qpos, axis=0)
    combined_qvel = np.concatenate(all_qvel, axis=0)
    combined_action = np.concatenate(all_action, axis=0)
    
    print(f"\nCombined dataset:")
    print(f"Total episodes: {len(all_qpos)}")
    print(f"Total timesteps: {len(combined_qpos)}")
    print(f"Combined QPos shape: {combined_qpos.shape}")
    print(f"Combined Action shape: {combined_action.shape}")
    
    # Print episode breakdown
    print(f"\nEpisode breakdown:")
    for info in episode_info:
        print(f"  {info['filename']}: {info['length']} steps (indices {info['start_idx']}-{info['end_idx']-1})")
    
    return combined_qpos, combined_qvel, combined_action, all_image_dicts

# Load all episodes from the directory
data_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task1c'
qpos, qvel, action, image_dict_list = load_all_episodes(data_dir)

if qpos is None:
    print("Failed to load any data!")
    exit()

# Create DataFrames with joint headers
joint_headers = ['timestamp', 'b', 's', 'e', 't', 'r', 'g']

# Create timestamp column (assuming sequential timesteps across all episodes)
num_timesteps = len(qpos)
timestamps = np.arange(num_timesteps)

# Create qpos DataFrame
qpos_data = np.column_stack([timestamps, qpos])
qpos_df = pd.DataFrame(qpos_data, columns=joint_headers)

# Create action DataFrame
action_data = np.column_stack([timestamps, action])
action_df = pd.DataFrame(action_data, columns=joint_headers)

print("\nDataFrame Summary:")
print("QPos DataFrame shape:", qpos_df.shape)
print("Action DataFrame shape:", action_df.shape)
print("\nQPos DataFrame head:")
print(qpos_df.head())
print("\nAction DataFrame head:")
print(action_df.head())

# Check the data ranges for each joint
print("\nQPos data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {qpos_df[col].min():.3f} to {qpos_df[col].max():.3f}")

print("\nAction data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {action_df[col].min():.3f} to {action_df[col].max():.3f}")

# Optional: Plot data distribution across episodes
plt.figure(figsize=(15, 10))

joint_cols = ['b', 's', 'e', 't', 'r', 'g']
joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    plt.subplot(2, 3, i+1)
    plt.plot(qpos_df['timestamp'], qpos_df[joint], 'b-', alpha=0.7, label=f'QPos {joint}')
    plt.plot(action_df['timestamp'], action_df[joint], 'r-', alpha=0.7, label=f'Action {joint}')
    plt.title(f'{joint_name} Joint ({joint}) - All Episodes')
    plt.xlabel('Timestep')
    plt.ylabel('Joint Value (radians)')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.suptitle('Joint Data Across All Episodes', fontsize=16)
plt.tight_layout()
plt.show()

print("\nData loading complete! Now you can proceed with preprocessing and training.")

# 1.2 Check the data

In [ ]:
# Complete the normalization code
joint_cols = ['b', 's', 'e', 't', 'r', 'g']

qpos_normalized = {}
action_normalized = {}

for joint in joint_cols:
    # QPos processing
    qpos_joint = qpos_df[joint].fillna(method="ffill")
    qpos_joint = np.array(qpos_joint)
    
    # Simple differences (like velocities)
    qpos_diff = np.diff(qpos_joint)
    qpos_csum_diff = qpos_diff.cumsum()
    
    # Z-score normalization of differences
    qpos_diff_normalized = (qpos_diff - np.mean(qpos_diff)) / (np.std(qpos_diff) + 1e-8)
    
    qpos_normalized[f'{joint}_diff'] = np.concatenate([[0], qpos_diff])
    qpos_normalized[f'{joint}_csum_diff'] = np.concatenate([[0], qpos_csum_diff])
    qpos_normalized[f'{joint}_diff_norm'] = np.concatenate([[0], qpos_diff_normalized])
    
    # Action processing
    action_joint = action_df[joint].fillna(method="ffill")
    action_joint = np.array(action_joint)
    
    action_diff = np.diff(action_joint)
    action_csum_diff = action_diff.cumsum()
    action_diff_normalized = (action_diff - np.mean(action_diff)) / (np.std(action_diff) + 1e-8)
    
    action_normalized[f'{joint}_diff'] = np.concatenate([[0], action_diff])
    action_normalized[f'{joint}_csum_diff'] = np.concatenate([[0], action_csum_diff])
    action_normalized[f'{joint}_diff_norm'] = np.concatenate([[0], action_diff_normalized])

# Create DataFrames from normalized data
qpos_norm_df = pd.DataFrame(qpos_normalized)
qpos_norm_df.insert(0, 'timestamp', qpos_df['timestamp'])

action_norm_df = pd.DataFrame(action_normalized)
action_norm_df.insert(0, 'timestamp', action_df['timestamp'])

print("Normalization completed!")
print(f"QPos normalized shape: {qpos_norm_df.shape}")
print(f"Action normalized shape: {action_norm_df.shape}")



# 2. Model Definition

## 2.1 Positional Encoding Layer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

In [ ]:
class ActionChunkingTransformer(nn.Module):
    def __init__(self, 
                 input_dim=6,        # 6 joint positions
                 output_dim=6,       # 6 joint actions
                 chunk_size=10,      # Predict next 10 actions
                 d_model=256,        # Embedding dimension
                 nhead=8,            # Attention heads
                 num_layers=6,       # Transformer layers
                 dropout=0.1):
        super().__init__()
        
        self.chunk_size = chunk_size
        self.output_dim = output_dim
        self.d_model = d_model
        
        # Input embedding: map joint positions to d_model dimensions
        self.input_embedding = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer encoder (processes state history)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=False  # [seq_len, batch, features]
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # ONLY ONE output head for action chunking
        self.action_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, output_dim * chunk_size)
        )
        
        # Causal mask for encoder (optional but recommended)
        self.register_buffer('causal_mask', None)
        
        self.init_weights()
    
    def init_weights(self):
        initrange = 0.1
        self.input_embedding.weight.data.uniform_(-initrange, initrange)
        for layer in self.action_head:
            if isinstance(layer, nn.Linear):
                layer.weight.data.uniform_(-initrange, initrange)
    
    def _generate_square_subsequent_mask(self, sz):
        """Generate causal mask to prevent looking at future timesteps"""
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask
    
    def forward(self, state_sequence):
        """
        Args:
            state_sequence: [seq_len, batch, input_dim] - state history: [state_t0, state_t1, state_t2, state_t3]
        Returns:
            action_chunk: [chunk_size, batch, output_dim] - future actions: [action_t4, action_t5, action_t6]
        """
        seq_len = state_sequence.size(0)
        batch_size = state_sequence.size(1)
        
        # Generate causal mask (optional for action chunking)
        if self.causal_mask is None or self.causal_mask.size(0) != seq_len:
            self.causal_mask = self._generate_square_subsequent_mask(seq_len).to(state_sequence.device)
        
        # 1. Embed state sequence
        embedded = self.input_embedding(state_sequence)  # [seq_len, batch, d_model]
        
        # 2. Add positional encoding
        with_pos = self.pos_encoder(embedded)  # [seq_len, batch, d_model]
        
        # 3. Process through transformer encoder
        encoded = self.transformer_encoder(with_pos, self.causal_mask)  # [seq_len, batch, d_model]
        
        # 4. Use ONLY the last timestep (current state) for prediction
        current_state_repr = encoded[-1]  # [batch, d_model]
        '''
        transformer → [encoded_t0, encoded_t1, encoded_t2, encoded_t3]
                                                                    ↑
                                                            Use only this (current state)
        '''
        
        # 5. Predict future action chunk
        future_actions_flat = self.action_head(current_state_repr)  # [batch, output_dim * chunk_size]
        # [action_t4, action_t5, action_t6] (future)
        
        # 6. Reshape to action sequence
        action_chunk = future_actions_flat.view(batch_size, self.chunk_size, self.output_dim)
        # Shape: [batch, chunk_size, output_dim]
        
        # 7. Transpose to [chunk_size, batch, output_dim] for consistency
        return action_chunk.transpose(0, 1)  # [chunk_size, batch, output_dim]

## 3.1. Create future window sequence

In [ ]:
# Future prediction
'''
Current State Representation → Optimal Future Action Sequence
# Training pairs
Input:  [states_t0_to_t100]     # 100 timesteps of state history
Target: [actions_t101_to_t110]   # Next 10 actions to execute

# The model learns: "Given this state history, these are the best next actions"
history_len = "state history" = 100
chunk_size = "action chunk" = 10
'''
def create_chunking_sequences(qpos_data, action_data, history_len, chunk_size):
    sequences = []
    
    for i in range(len(qpos_data) - history_len - chunk_size):
        state_history = qpos_data[i:i+history_len]                    # [t0, t1, t2, t3...t99]
        future_actions = action_data[i+history_len:i+history_len+chunk_size]  # [t100, t101, t102...t109]
        # Model learns: state_history → future_actions
        sequences.append((state_history, future_actions))
    return sequences

### 3.2. Split Data
Split data into the train and testing set, prepared in windowed sequences and pass through GPU

In [ ]:
def get_action_chunking_data(qpos_df, action_df, history_len=10, chunk_size=10, train_split=0.8):
    """
    Prepare robot data for action chunking training
    Args:
        qpos_df: DataFrame with joint positions
        action_df: DataFrame with joint actions
        history_len: length of state history sequence
        chunk_size: number of future actions to predict
        train_split: fraction of data for training
    Returns:
        train_data: list of (state_history, future_actions) tuples for training
        test_data: list of (state_history, future_actions) tuples for testing
        normalization_params: parameters for denormalizing predictions
    """
    # Extract joint columns (skip timestamp)
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    qpos_data = qpos_df[joint_cols].values  # [timesteps, 6 joints], timesteps should be history_len = 100
    action_data = action_df[joint_cols].values  # [timesteps, 6 joints] , timesteps should be chunk_size = 10

    # Normalize data (important for stable training)
    # Min-max normalization
    qpos_min, qpos_max = qpos_data.min(axis=0), qpos_data.max(axis=0)
    qpos_normalized = (qpos_data - qpos_min) / (qpos_max - qpos_min + 1e-8)
    
    action_min, action_max = action_data.min(axis=0), action_data.max(axis=0)
    action_normalized = (action_data - action_min) / (action_max - action_min + 1e-8)
    
    # Create action chunking sequences
    sequences = create_chunking_sequences(qpos_normalized, action_normalized, history_len, chunk_size)
    
    # Split into train/test
    split_idx = int(len(sequences) * train_split)
    train_sequences = sequences[:split_idx]
    test_sequences = sequences[split_idx:]
    
    # Convert to tensors
    train_data = [(torch.FloatTensor(seq[0]), torch.FloatTensor(seq[1])) 
                  for seq in train_sequences]
    test_data = [(torch.FloatTensor(seq[0]), torch.FloatTensor(seq[1])) 
                 for seq in test_sequences]
    
    # Store normalization parameters for later use
    normalization_params = {
        'qpos_min': qpos_min, 'qpos_max': qpos_max,
        'action_min': action_min, 'action_max': action_max
    }
    
    print(f"Action Chunking Data Preparation:")
    print(f"State history length: {history_len}")
    print(f"Action chunk size: {chunk_size}")
    print(f"Total sequences: {len(sequences)}")
    print(f"Training sequences: {len(train_data)}")
    print(f"Testing sequences: {len(test_data)}")
    print(f"Input shape per sequence: [history_len={history_len}, joints=6]")
    print(f"Output shape per sequence: [chunk_size={chunk_size}, joints=6]")
    
    return train_data, test_data, normalization_params

def get_action_chunking_batch(sequences, start_idx, batch_size):
    """
    Create batches for action chunking data
    Args:
        sequences: list of (state_history, future_actions) tuples
        start_idx: starting index
        batch_size: batch size
    Returns:
        state_histories: [history_len, batch, 6] - batched state sequences
        future_actions: [chunk_size, batch, 6] - batched future action chunks
    """
    end_idx = min(start_idx + batch_size, len(sequences))
    batch_sequences = sequences[start_idx:end_idx]
    
    # Stack sequences into batches
    state_histories = torch.stack([seq[0] for seq in batch_sequences], dim=1)  # [history_len, batch, 6]
    future_actions = torch.stack([seq[1] for seq in batch_sequences], dim=1)   # [chunk_size, batch, 6]
    
    return state_histories, future_actions

# Usage example:
history_len = 100  # Use 100 timesteps of state history
chunk_size = 10   # Predict next 10 actions

train_data, test_data, norm_params = get_action_chunking_data(
    qpos_df, action_df, 
    history_len=history_len,
    chunk_size=chunk_size,
    train_split=0.8
)

## 3.4. Set Model Parameter

In [ ]:
# Initialize transformer model with 256-dimensional embeddings, 8 heads for 
# Model parameters (make sure these match your data preparation)
model = ActionChunkingTransformer(
    input_dim=6,        # 6 joint positions
    output_dim=6,       # 6 joint actions  
    chunk_size=10,      # Predict next 10 actions
    d_model=256,        # Embedding dimension
    nhead=8,            # Attention heads
    num_layers=6,       # Transformer layers
    dropout=0.1
).to(device)

criterion = nn.MSELoss()
lr = 0.00005
epochs = 2000
batch_size = 32
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)

## 3.5. Training function

In [ ]:
def train_action_chunking_model(model, train_data, val_data, device, epochs=200, batch_size=32):
    """
    Train the action chunking transformer model
    """    
    train_losses = []
    val_losses = []
    
    print(f"Training action chunking transformer on {device}")
    print(f"Train sequences: {len(train_data)}, Val sequences: {len(val_data)}")
    print(f"Batch size: {batch_size}")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_train_loss = 0.0
        num_batches = 0
        start_time = time.time()
        
        for i in range(0, len(train_data), batch_size):
            # Get batch: state_histories, future_actions
            state_histories, future_actions = get_action_chunking_batch(train_data, i, batch_size)
            state_histories, future_actions = state_histories.to(device), future_actions.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass: predict future actions from state history
            predicted_actions = model(state_histories)  # [chunk_size, batch, 6]
            
            # Calculate loss
            loss = criterion(predicted_actions, future_actions)
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_train_loss += loss.item()
            num_batches += 1
            
            # Log progress within epoch
            if num_batches % max(1, len(train_data) // (batch_size * 5)) == 0:
                elapsed = time.time() - start_time
                avg_loss = total_train_loss / num_batches
                print(f"  Epoch {epoch:3d} | Batch {num_batches:4d}/{len(train_data)//batch_size:4d} | "
                      f"Loss: {avg_loss:.6f} | Time: {elapsed:.1f}s")
        
        avg_train_loss = total_train_loss / num_batches
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        total_val_loss = 0.0
        num_val_batches = 0
        
        with torch.no_grad():
            for i in range(0, len(val_data), batch_size):
                state_histories, future_actions = get_action_chunking_batch(val_data, i, batch_size)
                state_histories, future_actions = state_histories.to(device), future_actions.to(device)
                
                predicted_actions = model(state_histories)
                loss = criterion(predicted_actions, future_actions)
                
                total_val_loss += loss.item()
                num_val_batches += 1
        
        avg_val_loss = total_val_loss / num_val_batches if num_val_batches > 0 else float('inf')
        val_losses.append(avg_val_loss)
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)

        # Print epoch summary
        print(f"Epoch {epoch:3d} | Train Loss: {avg_train_loss:.6f} | "
              f"Val Loss: {avg_val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        print("-" * 80)
    
    return train_losses, val_losses

# Usage in your notebook:
print(f"Using device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train the model
train_losses, val_losses = train_action_chunking_model(
    model, train_data, test_data, device, epochs=epochs, batch_size=batch_size
)

# Plot training progress
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Action Chunking Training Progress')
plt.legend()
plt.yscale('log')

plt.subplot(1, 2, 2)
plt.plot(train_losses[-50:], label='Train Loss (Last 50)')
plt.plot(val_losses[-50:], label='Val Loss (Last 50)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Recent Training Progress')
plt.legend()

plt.tight_layout()
plt.show()

## 3.6. Evaluation function

In [ ]:
def evaluate_action_chunking_model_sequential_chunks(model, val_data, norm_params, device, num_chunks=5):
    """
    Evaluate action chunking model using sequential non-overlapping chunks from dataset
    Args:
        model: trained action chunking transformer model
        val_data: validation sequences (state_history, future_actions) tuples
        norm_params: normalization parameters for denormalizing predictions
        device: torch device
        num_chunks: number of sequential chunks to evaluate
    Returns:
        avg_loss: average validation loss
        concatenated_predictions: all predictions concatenated [total_steps, 6]
        concatenated_actuals: all actuals concatenated [total_steps, 6]
        timestamps: corresponding timestamps for plotting
    """
    model.eval()
    criterion = nn.MSELoss()
    
    joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    # Storage for results
    all_chunk_predictions = []
    all_chunk_actuals = []
    all_chunk_losses = []
    chunk_timestamps = []
    
    print(f"Evaluating {num_chunks} sequential chunks from dataset")
    print(f"Model chunk size: {model.chunk_size}")
    print(f"Each chunk: 100 timesteps → predict next {model.chunk_size} actions")
    print("=" * 60)
    
    with torch.no_grad():
        # Select sequential chunks from validation data
        chunk_spacing = max(1, len(val_data) // (num_chunks + 1))  # Space chunks evenly
        
        for chunk_idx in range(num_chunks):
            # Calculate the sequence index for this chunk
            seq_idx = chunk_idx * chunk_spacing
            
            if seq_idx >= len(val_data):
                print(f"Warning: Not enough sequences for {num_chunks} chunks")
                break
            
            print(f"Evaluating chunk {chunk_idx + 1}/{num_chunks} (sequence {seq_idx})...")
            
            # Get one sequence: (state_history, future_actions)
            state_history, future_actions = val_data[seq_idx]
            
            # Prepare input: add batch dimension
            state_input = state_history.unsqueeze(1).to(device)  # [history_len, 1, 6]
            future_target = future_actions.to(device)  # [chunk_size, 6]
            
            # Predict future actions from state history
            predicted_chunk = model(state_input)  # [chunk_size, 1, 6]
            predicted_chunk = predicted_chunk.squeeze(1)  # [chunk_size, 6]
            
            # Store predictions and actuals for this chunk
            all_chunk_predictions.append(predicted_chunk.cpu())  # [chunk_size, 6]
            all_chunk_actuals.append(future_target.cpu())        # [chunk_size, 6]
            
            # Calculate loss for this chunk
            chunk_loss = criterion(predicted_chunk, future_target)
            all_chunk_losses.append(chunk_loss.item())
            
            # Create timestamps for this chunk (assuming 100 + chunk_size timesteps per sequence)
            chunk_start_time = seq_idx * (100 + model.chunk_size) + 100  # Start after the 100 input timesteps
            chunk_times = list(range(chunk_start_time, chunk_start_time + model.chunk_size))
            chunk_timestamps.extend(chunk_times)
            
            print(f"  Chunk {chunk_idx + 1}: Loss = {chunk_loss:.6f}, Timestamps = {chunk_start_time}-{chunk_start_time + model.chunk_size - 1}")
    
    if not all_chunk_predictions:
        print("No chunks were evaluated!")
        return float('inf'), None, None, None, None, None
    
    # Concatenate all chunks into single sequences
    concatenated_predictions = torch.cat(all_chunk_predictions, dim=0)  # [total_steps, 6]
    concatenated_actuals = torch.cat(all_chunk_actuals, dim=0)          # [total_steps, 6]
    
    # Calculate overall average loss
    avg_loss = np.mean(all_chunk_losses)
    
    print(f"\nOverall Results:")
    print(f"Total prediction steps: {len(concatenated_predictions)}")
    print(f"Average loss across all chunks: {avg_loss:.6f}")
    
    # Denormalize predictions and actuals
    if 'action_min' in norm_params and 'action_max' in norm_params:
        # Min-max denormalization: value = normalized * (max - min) + min
        action_min = torch.FloatTensor(norm_params['action_min'])
        action_max = torch.FloatTensor(norm_params['action_max'])
        action_range = action_max - action_min
        
        # Denormalize concatenated results
        denorm_predictions = concatenated_predictions * action_range + action_min
        denorm_actuals = concatenated_actuals * action_range + action_min
        
        # Convert to numpy for plotting
        denorm_predictions = denorm_predictions.numpy()
        denorm_actuals = denorm_actuals.numpy()
    
    else:
        # Fallback: use normalized values if no proper normalization params found
        print("Warning: No proper normalization parameters found. Using normalized values.")
        denorm_predictions = concatenated_predictions.numpy()
        denorm_actuals = concatenated_actuals.numpy()
    
    return avg_loss, denorm_predictions, denorm_actuals, chunk_timestamps, joint_names, joint_cols


def plot_sequential_chunks_results(predictions, actuals, timestamps, joint_names, joint_cols):
    """
    Create comprehensive plots for sequential chunks evaluation results
    """
    if predictions is None or actuals is None:
        print("No data to plot!")
        return
    
    total_steps = len(predictions)
    
    # Plot 1: Time series for all joints showing concatenated predictions
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()
    
    for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
        axes[i].plot(timestamps, actuals[:, i], 'b-o', linewidth=2, 
                    label=f'Actual {joint_col}', alpha=0.8, markersize=4)
        axes[i].plot(timestamps, predictions[:, i], 'r--s', linewidth=2, 
                    label=f'Predicted {joint_col}', alpha=0.8, markersize=4)
        
        axes[i].set_title(f'{joint_name} Joint ({joint_col}) - Sequential Chunks')
        axes[i].set_xlabel('Timestamp')
        axes[i].set_ylabel('Joint Action Value')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
        
        # Calculate and display error metrics
        mse = np.mean((actuals[:, i] - predictions[:, i])**2)
        mae = np.mean(np.abs(actuals[:, i] - predictions[:, i]))
        
        # Add vertical lines to show chunk boundaries (every 10 steps)
        chunk_size = 10  # Assuming chunk_size = 10
        for chunk_boundary in range(chunk_size, total_steps, chunk_size):
            if chunk_boundary < len(timestamps):
                axes[i].axvline(x=timestamps[chunk_boundary], color='gray', linestyle=':', alpha=0.5)
        
        axes[i].text(0.05, 0.95, f'MSE: {mse:.4f}\nMAE: {mae:.4f}', 
                    transform=axes[i].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.suptitle(f'Sequential Action Chunk Predictions - {total_steps} Total Steps', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Plot 2: Error analysis by chunk position
    chunk_size = 10
    num_chunks = total_steps // chunk_size
    
    if num_chunks > 1:
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        axes = axes.flatten()
        
        for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
            chunk_errors = []
            chunk_labels = []
            
            for chunk_idx in range(num_chunks):
                start_idx = chunk_idx * chunk_size
                end_idx = min((chunk_idx + 1) * chunk_size, total_steps)
                
                if start_idx < total_steps:
                    chunk_actual = actuals[start_idx:end_idx, i]
                    chunk_pred = predictions[start_idx:end_idx, i]
                    chunk_mae = np.mean(np.abs(chunk_actual - chunk_pred))
                    
                    chunk_errors.append(chunk_mae)
                    chunk_labels.append(f'Chunk {chunk_idx + 1}')
            
            if chunk_errors:
                axes[i].bar(range(len(chunk_errors)), chunk_errors, alpha=0.7)
                axes[i].set_title(f'{joint_name} Joint ({joint_col}) - Error by Chunk')
                axes[i].set_xlabel('Chunk Number')
                axes[i].set_ylabel('Mean Absolute Error')
                axes[i].set_xticks(range(len(chunk_labels)))
                axes[i].set_xticklabels(chunk_labels, rotation=45)
                axes[i].grid(True, alpha=0.3)
        
        plt.suptitle('Prediction Error Analysis by Chunk', fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Plot 3: Correlation analysis
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
        actual_vals = actuals[:, i]
        pred_vals = predictions[:, i]
        
        axes[i].scatter(actual_vals, pred_vals, alpha=0.6, s=20)
        
        # Perfect prediction line
        min_val = min(actual_vals.min(), pred_vals.min())
        max_val = max(actual_vals.max(), pred_vals.max())
        axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Perfect Prediction')
        
        # Calculate correlation
        correlation = np.corrcoef(actual_vals, pred_vals)[0, 1]
        
        axes[i].set_title(f'{joint_name} Joint ({joint_col})\nCorr: {correlation:.3f}')
        axes[i].set_xlabel('Actual Actions')
        axes[i].set_ylabel('Predicted Actions')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
    
    plt.suptitle('Predicted vs Actual Actions Correlation Analysis', fontsize=16)
    plt.tight_layout()
    plt.show()


def comprehensive_sequential_chunks_evaluation(model, val_data, norm_params, device, num_chunks=5):
    """
    Complete evaluation pipeline for sequential action chunks
    """
    print("Starting comprehensive sequential chunks evaluation...")
    print("=" * 80)
    
    # Evaluate model
    avg_loss, predictions, actuals, timestamps, joint_names, joint_cols = evaluate_action_chunking_model_sequential_chunks(
        model, val_data, norm_params, device, num_chunks=num_chunks
    )
    
    if predictions is None:
        print("Evaluation failed!")
        return None, None, None
    
    print(f"\nAverage Validation Loss: {avg_loss:.6f}")
    
    # Calculate overall statistics
    all_errors = []
    joint_errors = {joint: [] for joint in joint_cols}
    
    for i in range(len(predictions)):
        for joint_idx, joint in enumerate(joint_cols):
            error = abs(actuals[i, joint_idx] - predictions[i, joint_idx])
            all_errors.append(error)
            joint_errors[joint].append(error)
    
    print(f"\nOverall Statistics:")
    print(f"Total timesteps evaluated: {len(predictions)}")
    print(f"Mean Absolute Error: {np.mean(all_errors):.4f}")
    print(f"Max Absolute Error: {np.max(all_errors):.4f}")
    print(f"Std Absolute Error: {np.std(all_errors):.4f}")
    
    print(f"\nPer-Joint Statistics:")
    for joint in joint_cols:
        if joint_errors[joint]:
            print(f"Joint {joint}: MAE = {np.mean(joint_errors[joint]):.4f}, "
                  f"Max = {np.max(joint_errors[joint]):.4f}")
    
    # Create plots
    plot_sequential_chunks_results(predictions, actuals, timestamps, joint_names, joint_cols)
    
    return avg_loss, predictions, actuals

# Usage after training your action chunking model:
avg_loss, predictions, actuals = comprehensive_sequential_chunks_evaluation(
    model, test_data, norm_params, device, num_chunks=5
)